# Vision · 범위 제한 기반 빈칸·부분 구현 문제

원본에 마스킹이 없어 **완성된 실습 코드를 바탕으로 새로 선정한 연습 문제**입니다. 실제 시험 문항이나 공식 배점을 의미하지 않습니다.

- ResNet: **0~5절**, 전체 셀15까지. 이번 요청의 “Chapter 6 이전”을 적용하여 6절 자체도 제외합니다.
- ViT: **6절 시작 전**, 전체 셀23까지. 원본 절3/4 순서가 뒤섞여 있어 셀 번호도 함께 기록합니다.
- DETR: 기존 확정 범위인 **13절 포함**, 전체 셀30까지.
- U-Net: 지정 노트북 전체. 설치·다운로드·긴 학습·그래프 꾸미기는 제외합니다.

근거: [시험 출제 포인트 PDF](../../AI인증시험_출제포인트.pdf) 1쪽 Vision의 레이어 이해, 예측/정답을 학습으로 연결, 데이터 구성, 이미지 변환 라이브러리, 평가 Metric. 문항별 셀 번호는 **Markdown 포함,1부터**입니다.

**사용법:** 준비 셀을 먼저 실행하고 각 문제에서 `MISSING` 또는 `raise NotImplementedError`를 올바른 코드로 교체 → 바로 아래 테스트 실행 → 실패하면 shape/축/순서 확인 → 마지막의 접힌 해설 확인. `# ????`는 원본에서 가져온 빈칸이 아니라 새 마스킹 표시입니다. 각 문제는 독립되어 이전 문제의 답이 필요 없습니다. 답을 수정하면 함수 셀을 먼저 다시 실행하세요. 예상 5~10분/문제(연습용 가안). 작성한 이 노트북을 제출하면 채점할 수 있습니다.

CPU·PyTorch·torchvision·einops·NumPy·Pillow만 사용합니다. 모델/데이터 다운로드 없음. 테스트는 shape뿐 아니라 수치, 학습 갱신, 잘못된 축, 다른 인자도 검사합니다. 통과는 작은 입력 검증이며 실전 정확도나 독립 숙달을 보장하지 않습니다.

**원본과 차이:** ViT FFN은 실제 코드의 ReLU 사용(주석의 GELU와 다름). U-Net DoubleConv는 실제 코드처럼 BN 없음. DETR attention reshape는 query/key 축이 뒤바뀐 설명·슬라이스를 교정하고 문제에서 명시합니다. ResNet AMP는 CPU에서 비활성 scaler로 순서를 확인합니다. 설치된 PyTorch 2.5.1/torchvision 0.20.1/einops 0.8.2에서 검증합니다.

이미지 텐서는 PyTorch의 실제 축 순서 **B,C,H,W**로 표기합니다. W,H를 구분하는 비정사각형 입력도 검사합니다. 이해가 안 되는 문제 번호·줄을 알려주면 작은 행렬 예시로 다시 설명할 수 있습니다.


In [ ]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

torch.set_num_threads(1)
MISSING = object()  # ???? 대신 실행 가능한 표식. 이 값을 올바른 코드로 바꾸세요.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

def close(actual, expected):
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)

class ModeProbe(nn.Module):
    # train/eval과 gradient 모드를 기록하는 작은 제공 분류기입니다.
    def __init__(self, width=4, classes=3):
        super().__init__()
        self.fc = nn.Linear(width, classes)
        self.seen = []
    def forward(self, x):
        self.seen.append((self.training, torch.is_grad_enabled()))
        return self.fc(x)


## D01. DETR 입력 리사이즈·정규화·배치

**PDF 점검:** 데이터 구성·이미지 변환 · **근거:** [03_DETR.ipynb](../../vision/03_DETR.ipynb) 전체 셀 8, 18

PIL RGB 이미지를 짧은 변 short_side에 맞춰 종횡비를 유지한 Resize→ToTensor→ImageNet Normalize 후 batch축을 추가하세요. 실제 원본은800이지만 테스트는8/10입니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_d01(image,short_side):
    transform=T.Compose([
        MISSING,
        T.ToTensor(),
        MISSING,
    ])
    return MISSING

In [ ]:
def check_d01(fn):
    for side in (8,10):
        im=Image.fromarray(np.full((6,12,3),128,dtype=np.uint8))
        got=fn(im,side)
        assert got.shape==(1,3,side,2*side)
        for c,(mean,std) in enumerate(zip([.485,.456,.406],[.229,.224,.225])):
            close(got[0,c],torch.full((side,2*side),(128/255-mean)/std))

try:
    check_d01(solve_d01)
    print("PASS · D01")
except Exception as error:
    print("FAIL · D01 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: H6,W12에 Resize(8)→H8,W16。Resize((8,8))은 정사각형 강제 변환이라 다릅니다. 최종 입력은(1,3,8,16)입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_d01(image,short_side):
    transform=T.Compose([
        T.Resize(short_side),
        T.ToTensor(),
        T.Normalize([.485,.456,.406],[.229,.224,.225]),
    ])
    return transform(image).unsqueeze(0)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## D02. 중심 박스를 픽셀 꼭짓점으로

**PDF 점검:** 데이터 변환·모델 출력 활용 · **근거:** [03_DETR.ipynb](../../vision/03_DETR.ipynb) 전체 셀 8

boxes=(N,4)는 정규화된 cx,cy,w,h입니다. xyxy로 변환하고 PIL size=(W,H)를 이용해 픽셀 좌표로 되돌리세요. 빈 박스 N=0도 처리해야 합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_d02(boxes,size):
    cx,cy,w,h=boxes.unbind(1)
    corners=MISSING
    img_w,img_h=size
    return MISSING

In [ ]:
def check_d02(fn):
    boxes=torch.tensor([[.5,.5,.2,.4],[.25,.25,.5,.5]])
    close(fn(boxes,(200,100)),torch.tensor([[80.,30.,120.,70.],[0.,0.,100.,50.]]))
    close(fn(boxes,(100,200)),torch.tensor([[40.,60.,60.,140.],[0.,0.,50.,100.]]))
    assert fn(torch.empty(0,4),(200,100)).shape==(0,4)

try:
    check_d02(solve_d02)
    print("PASS · D02")
except Exception as error:
    print("FAIL · D02 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: cx=.5,cy=.5,w=.2,h=.4→xyxy=[.4,.3,.6,.7]。이미지 W200,H100이면[80,30,120,70]입니다. 텐서는H,W 순서지만 PIL size는W,H 순서입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_d02(boxes,size):
    cx,cy,w,h=boxes.unbind(1)
    corners=torch.stack([cx-.5*w,cy-.5*h,cx+.5*w,cy+.5*h],dim=1)
    img_w,img_h=size
    return corners * boxes.new_tensor([img_w,img_h,img_w,img_h])
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## D03. no-object 제외와 confidence 필터

**PDF 점검:** 모델 출력 활용·데이터 구성 · **근거:** [03_DETR.ipynb](../../vision/03_DETR.ipynb) 전체 셀 18

outputs의 pred_logits=(1,Q,C+1),pred_boxes=(1,Q,4)를 받습니다. 전체 C+1에 softmax 후 마지막 no-object 열을 제외하고, 최고 객체 확률이 threshold보다 큰 query만 남기세요. 반환은 선택된 객체확률,선택된 정규화박스,keep 마스크입니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_d03(outputs,threshold):
    probas=MISSING
    keep=MISSING
    return probas[keep],MISSING,keep

In [ ]:
def check_d03(fn):
    probs=torch.tensor([[[.95,.02,.03],[.03,.02,.95],[.4,.5,.1]]])
    boxes=torch.arange(12).float().reshape(1,3,4)/12
    outputs={'pred_logits':probs.log(),'pred_boxes':boxes}
    for threshold,ids in [(.9,[0]),(.45,[0,2]),(.99,[])]:
        p,b,keep=fn(outputs,threshold)
        expected=torch.tensor([i in ids for i in range(3)])
        close(keep,expected);close(p,probs[0,expected,:-1]);close(b,boxes[0,expected])
        assert keep.dtype==torch.bool

try:
    check_d03(solve_d03)
    print("PASS · D03")
except Exception as error:
    print("FAIL · D03 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: [고양이.03,개.02,no-object.95]인 query는 버립니다. no-object를 먼저 제거한 뒤 softmax하면 .6/.4로 재정규화되어 객체 확률을 부풀릴 수 있습니다. confidence threshold는 IoU threshold나 mAP가 아닙니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_d03(outputs,threshold):
    probas=outputs['pred_logits'].softmax(-1)[0,:,:-1]
    keep=probas.max(-1).values > threshold
    return probas[keep],outputs['pred_boxes'][0,keep],keep
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## D04. forward hook으로 attention 추출

**PDF 점검:** 레이어 활용 · **근거:** [03_DETR.ipynb](../../vision/03_DETR.ipynb) 전체 셀 22

제공 모듈은 forward에서 (결과,attention_weights)를 반환합니다. hook으로 output[1]을 목록에 저장하고 forward 1회 후 hook을 제거하세요. 반환 saved[0]은 가중치 텐서입니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_d04(module,x):
    saved=[]
    handle=module.register_forward_hook(MISSING)
    module(x)
    raise NotImplementedError("???? 이 줄을 완성하세요")
    return saved[0]

In [ ]:
def check_d04(fn):
    class ToyAttention(nn.Module):
        def forward(self,x):return x*2,x@x.transpose(-1,-2)
    for n in (3,5):
        m=ToyAttention();x=torch.randn(1,n,2)
        close(fn(m,x),x@x.transpose(-1,-2))
        assert len(m._forward_hooks)==0
        m(x);assert len(m._forward_hooks)==0

try:
    check_d04(solve_d04)
    print("PASS · D04")
except Exception as error:
    print("FAIL · D04 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: forward가 (output=(1,3,2),weights=(1,3,3))를 반환하면 output[1]만 저장합니다. hook의 output 인자는 이 반환 튜플 전체입니다. 완료 후 remove해야 다음 실행 때 중복 저장되지 않습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_d04(module,x):
    saved=[]
    handle=module.register_forward_hook(lambda module,inputs,output: saved.append(output[1]))
    module(x)
    handle.remove()
    return saved[0]
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## D05. attention 토큰축을 공간으로 복원

**PDF 점검:** 레이어 동작·출력 활용 · **근거:** [03_DETR.ipynb](../../vision/03_DETR.ipynb) 전체 셀 24, 28, 30

encoder_attn=(1,HW,HW),decoder_attn=(1,Q,HW)를 제공합니다(헤드 평균 완료). encoder를(H,W,H,W)로 바꾸고, query 위치(qy,qx)가 보는 key지도(H,W)와 decoder query_id의 지도(H,W)를 반환하세요. 주의: 원본30셀의 마지막 두 축 슬라이스는 반대 방향입니다. 여기서는 query→key 의미에 맞게 앞 두 축을 고릅니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_d05(encoder_attn,decoder_attn,h,w,qy,qx,query_id):
    spatial=MISSING
    enc_map=MISSING
    dec_map=MISSING
    return enc_map,dec_map

In [ ]:
def check_d05(fn):
    for h,w in [(2,3),(3,2)]:
        s=h*w;enc=torch.arange(s*s).float().reshape(1,s,s);dec=torch.arange(4*s).float().reshape(1,4,s)
        a,b=fn(enc,dec,h,w,1,0,2)
        close(a,enc[0,w,:].reshape(h,w));close(b,dec[0,2,:].reshape(h,w))

try:
    check_d05(solve_d05)
    print("PASS · D05")
except Exception as error:
    print("FAIL · D05 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: H2,W3에서 query(y1,x0)는1×3+0=3번 토큰입니다. encoder[0,3,:]가 이 query가 모든 key를 보는 행입니다. 원본 설명의 query/key 축 표기가 뒤바뀐 부분을 보정했습니다. 실제 head별 출력이면 먼저 head 선택/평균이 필요합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_d05(encoder_attn,decoder_attn,h,w,qy,qx,query_id):
    spatial=encoder_attn[0].reshape(h,w,h,w)
    enc_map=spatial[qy,qx,:,:]
    dec_map=decoder_attn[0,query_id].reshape(h,w)
    return enc_map,dec_map
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>